# 一个更加哇塞的Attention

解决上一节中Attention存在的问题：

1、d_k要在__init__里定义。

2、Q、K、V三个矩阵太繁琐。

In [1]:
#上一节里我们最后得到了这样的一个Attention类
import torch
class Attention(torch.nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, Q, K, V):
        d_k = torch.tensor(Q.shape[-1])
        attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)
        attention_weights = torch.softmax(attention_weights/torch.sqrt(d_k), dim=-1)
        output = torch.einsum("bqL, bLd->bqd", attention_weights, V)
        return output

### 回顾一下attention的图

<img src='./figure/normal_attention.png' width="15%">

### 思考解决方案

1、d_k在__init__里好定义，就是一个参数的事

2、如果Attention接受的不是token而是处理好的feature的话，那么Attention的输入输出都长一样了，几个Attention也可以直连了

3、如果1和2都定义好了，那其实也就知道从输入token到Q K V的矩阵是什么样的了，那就可以把Q K V的怎么得来的方法也放在里，这里比较常见的就是用一个linear

In [2]:
#更改后的Attention
class Attention(torch.nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.d_k = dims
        
        self.q_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.k_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.v_proj = torch.nn.Linear(self.d_k, self.d_k)

    # 现在输入只有一个x了，Q K V都由x产生
    def forward(self, x):
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)
        attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)
        #d_k直接取参数
        attention_weights = torch.softmax(attention_weights/torch.sqrt(self.d_k), dim=-1)
        output = torch.einsum("bqL, bLd->bqd", attention_weights, V)
        return output

### 出现的新问题

这种形式下，如果输入x的shape是(batch, 1, dims)，那么Q K V的shape也都是(batch, 1, dims)，也就是说前面的缓存都没有了

这显然是不合理的，需要人为设定一项来充当缓存

In [3]:
class Attention(torch.nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.d_k = torch.tensor(dims)
        
        self.q_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.k_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.v_proj = torch.nn.Linear(self.d_k, self.d_k)

        self.k_cache = None
        self.v_cache = None

    def forward(self, x):
        Q = self.q_proj(x)

        # 如果都是空的那就直接赋值，不然就在sequence-length维度往后加
        if self.k_cache is None and self.v_cache is None:
            self.k_cache = self.k_proj(x)
            self.v_cache = self.v_proj(x)
        else:
            self.k_cache = torch.cat([self.k_cache, self.k_proj(x)], dim=1)
            self.v_cache = torch.cat([self.v_cache, self.v_proj(x)], dim=1)

        K = self.k_cache
        V = self.v_cache

        attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)
        attention_weights = torch.softmax(attention_weights/torch.sqrt(self.d_k), dim=-1)
        output = torch.einsum("bqL, bLd->bqd", attention_weights, V)
        return output
    
    #设定了cache就要有清除他的东西
    def emplty_kv_cache(self):
        self.k_cache = None
        self.v_cache = None

In [4]:
# 测试一下
batch_size = 2
dims = 16
attention_layer = Attention(dims=dims)
input_ids = torch.randn(batch_size, 1, dims)

In [5]:
attention_layer.emplty_kv_cache()
for i in range(5):
    output = attention_layer(input_ids)
    print(f"在产生{i}个token时，k_cache的shape是{attention_layer.k_cache.shape}")

在产生0个token时，k_cache的shape是torch.Size([2, 1, 16])
在产生1个token时，k_cache的shape是torch.Size([2, 2, 16])
在产生2个token时，k_cache的shape是torch.Size([2, 3, 16])
在产生3个token时，k_cache的shape是torch.Size([2, 4, 16])
在产生4个token时，k_cache的shape是torch.Size([2, 5, 16])


In [6]:
# 如果不进行kv_cache清零的话会出现上一轮的kv_cache还在累积
for i in range(5):
    output = attention_layer(input_ids)
    print(f"在产生{i}个token时，k_cache的shape是{attention_layer.k_cache.shape}")

在产生0个token时，k_cache的shape是torch.Size([2, 6, 16])
在产生1个token时，k_cache的shape是torch.Size([2, 7, 16])
在产生2个token时，k_cache的shape是torch.Size([2, 8, 16])
在产生3个token时，k_cache的shape是torch.Size([2, 9, 16])
在产生4个token时，k_cache的shape是torch.Size([2, 10, 16])


### 回顾一下整个transformer的流程

<img src='./figure/transformers.png' width="40%">

可以发现这个✖️N的东西里面除了Attention外，还有个前馈网络（Feed Forward）和残差连接和Norm

前馈网络就是一个MLP，现在前馈网络也搞很花哨，我们就简简单单一个升维再降维，激活函数选relu

这个Norm在transformer里选择的是LayerNorm。

为什么不用BatchNorm呢？

想一想图像一个卷积域内batch之间的信息可能有什么用？nlp里一句话里一个token内batch之间的信息可能有什么用？

再想一想一个句子里各个token之间的信息可能有什么用？

为什么用BatchNorm就很明确了。

In [7]:
# 给Attention加点东西
class Attention(torch.nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.d_k = torch.tensor(dims)
        
        self.q_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.k_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.v_proj = torch.nn.Linear(self.d_k, self.d_k)

        self.k_cache = None
        self.v_cache = None
        # 中间层直接设置成2*dims
        self.feed_forward = torch.nn.Sequential(
            torch.nn.Linear(dims, 2*dims),
            torch.nn.ReLU(),
            torch.nn.Linear(2*dims, dims)
        )

        self.norm = torch.nn.LayerNorm(dims)

    def forward(self, x):

        res = x
        Q = self.q_proj(x)
        if self.k_cache is None and self.v_cache is None:
            self.k_cache = self.k_proj(x)
            self.v_cache = self.v_proj(x)
        else:
            self.k_cache = torch.cat([self.k_cache, self.k_proj(x)], dim=1)
            self.v_cache = torch.cat([self.v_cache, self.v_proj(x)], dim=1)

        K = self.k_cache
        V = self.v_cache

        attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)
        attention_weights = torch.softmax(attention_weights/torch.sqrt(self.d_k), dim=-1)
        output = torch.einsum("bqL, bLd->bqd", attention_weights, V)
        #attention完先来个res+norm
        output = self.norm(output+res)
        #mlp完再来个res+norm
        res = output
        output = self.feed_forward(output)
        output = self.norm(output+res)
        
        return output

    def emplty_kv_cache(self):
        self.k_cache = None
        self.v_cache = None

In [8]:
# 测试一下
batch_size = 2
dims = 16
attention_layer = Attention(dims=dims)
input_ids = torch.randn(batch_size, 1, dims)

attention_layer.emplty_kv_cache()
for i in range(5):
    output = attention_layer(input_ids)
    print(f"在产生{i}个token时，k_cache的shape是{attention_layer.k_cache.shape}")

在产生0个token时，k_cache的shape是torch.Size([2, 1, 16])
在产生1个token时，k_cache的shape是torch.Size([2, 2, 16])
在产生2个token时，k_cache的shape是torch.Size([2, 3, 16])
在产生3个token时，k_cache的shape是torch.Size([2, 4, 16])
在产生4个token时，k_cache的shape是torch.Size([2, 5, 16])


###  Now 尝试搭建一个Model吧

现在看起来我们的Attention已经初具规模了，虽然还有一些不完善的地方，比如论文里的mask、multi-head都没有搞出来，

但是不影响，我们将在[第三节](./03.ipynb)里用这个Attention类搭建一个简单的可以做自回归任务的Model类，之后再进行完善。